In [ ]:
%pip install entsoe-py

In [ ]:
dbutils.library.restartPython()

In [ ]:
# Parameters
from datetime import date, timedelta

dbutils.widgets.text("start_date", "")
dbutils.widgets.text("end_date", "")

# empty widgets = last 7 days (ENTSO-E revises its data a few days after publication)
end_date = dbutils.widgets.get("end_date") or str(date.today())
start_date = dbutils.widgets.get("start_date") or str(date.today() - timedelta(days=7))

api_key = dbutils.secrets.get(catalog="energy", schema="configschema", key="ENTSOE_API_KEY")

# zones solved by flow tracing + neighbours whose mix we need for imports
MODELED_ZONES = ["BE", "FR", "NL", "DE_LU"]
EXTERNAL_ZONES = ["GB", "CH", "ES", "IT_NORD", "AT", "CZ", "PL", "DK_1", "DK_2", "NO_2", "SE_4"]

spark.conf.set("spark.sql.session.timeZone", "UTC")
print(start_date, "->", end_date)

In [ ]:
%sql
-- bronze schema and tables
USE CATALOG energy;
CREATE SCHEMA IF NOT EXISTS d1_bronze;

CREATE TABLE IF NOT EXISTS d1_bronze.bronze_generation (
  zone_code STRING,
  ts_utc TIMESTAMP,
  production_type STRING,
  generation_mw DOUBLE,
  ingested_at TIMESTAMP
) USING DELTA;

CREATE TABLE IF NOT EXISTS d1_bronze.bronze_load (
  zone_code STRING,
  ts_utc TIMESTAMP,
  load_mw DOUBLE,
  ingested_at TIMESTAMP
) USING DELTA;

CREATE TABLE IF NOT EXISTS d1_bronze.bronze_flows (
  from_zone STRING,
  to_zone STRING,
  ts_utc TIMESTAMP,
  flow_mw DOUBLE,
  ingested_at TIMESTAMP
) USING DELTA;

In [ ]:
# ENTSO-E client and borders
import pandas as pd
from entsoe import EntsoePandasClient
from entsoe.exceptions import NoMatchingDataError
from entsoe.mappings import NEIGHBOURS

client = EntsoePandasClient(api_key=api_key)
start = pd.Timestamp(start_date, tz="UTC")
end = pd.Timestamp(end_date, tz="UTC")

# physical flows are directional: we need both directions of every border touching a modeled zone
all_zones = MODELED_ZONES + EXTERNAL_ZONES
borders = set()
for zone in MODELED_ZONES:
    for neighbour in NEIGHBOURS[zone]:
        if neighbour in all_zones:
            borders.add((zone, neighbour))
            borders.add((neighbour, zone))
borders = sorted(borders)
print(len(borders), "directed borders")

In [ ]:
# write a pandas dataframe to bronze (the window is deleted first, so a re-run creates no duplicates)
def save_to_bronze(pdf, table, key):
    if pdf.empty:
        return
    pdf["ingested_at"] = pd.Timestamp.now(tz="UTC").tz_localize(None)
    spark.createDataFrame(pdf).createOrReplaceTempView("staging")
    spark.sql(f"""
        DELETE FROM {table}
        WHERE ts_utc >= '{start_date}' AND ts_utc < '{end_date}'
          AND {key} IN (SELECT DISTINCT {key} FROM staging)
    """)
    spark.sql(f"INSERT INTO {table} SELECT * FROM staging")


def to_utc(index):
    return pd.to_datetime(index, utc=True).tz_convert(None)

In [ ]:
# Generation per production type (bronze_generation)
frames = []
for zone in all_zones:
    try:
        gen = client.query_generation(zone, start=start, end=end)
    except NoMatchingDataError:
        print(zone, "no data")        # e.g. GB, not published on ENTSO-E since Brexit
        continue
    # some zones also report storage consumption: we only keep generation
    if isinstance(gen.columns, pd.MultiIndex):
        gen = gen.xs("Actual Aggregated", axis=1, level=1)
    gen.index = to_utc(gen.index)
    gen = gen.rename_axis("ts_utc").reset_index()
    gen = gen.melt(id_vars="ts_utc", var_name="production_type", value_name="generation_mw").dropna()
    gen["zone_code"] = zone
    frames.append(gen[gen.ts_utc < end.tz_convert(None)])
    print(zone, len(gen), "rows")

generation = pd.concat(frames)[["zone_code", "ts_utc", "production_type", "generation_mw"]]
save_to_bronze(generation, "d1_bronze.bronze_generation", "zone_code")

In [ ]:
# Actual total load (bronze_load)
frames = []
for zone in MODELED_ZONES:
    try:
        load = client.query_load(zone, start=start, end=end)
    except NoMatchingDataError:
        print(zone, "no data")
        continue
    frames.append(pd.DataFrame({
        "zone_code": zone,
        "ts_utc": to_utc(load.index),
        "load_mw": load["Actual Load"].values,
    }))
    print(zone, len(load), "rows")

load = pd.concat(frames).dropna()
save_to_bronze(load[load.ts_utc < end.tz_convert(None)], "d1_bronze.bronze_load", "zone_code")

In [ ]:
# Cross-border physical flows (bronze_flows)
frames = []
for from_zone, to_zone in borders:
    try:
        flow = client.query_crossborder_flows(from_zone, to_zone, start=start, end=end)
    except NoMatchingDataError:
        print(from_zone, "->", to_zone, "no data")
        continue
    frames.append(pd.DataFrame({
        "from_zone": from_zone,
        "to_zone": to_zone,
        "ts_utc": to_utc(flow.index),
        "flow_mw": flow.values,
    }))

flows = pd.concat(frames).dropna()
flows = flows[flows.ts_utc < end.tz_convert(None)]
print(len(flows), "flow rows")
save_to_bronze(flows, "d1_bronze.bronze_flows", "concat(from_zone, '>', to_zone)")

In [ ]:
%sql
-- bronze check
SELECT 'generation' AS source, zone_code, COUNT(*) AS n_rows, MIN(ts_utc) AS first_ts, MAX(ts_utc) AS last_ts
FROM energy.d1_bronze.bronze_generation GROUP BY zone_code
UNION ALL
SELECT 'load', zone_code, COUNT(*), MIN(ts_utc), MAX(ts_utc)
FROM energy.d1_bronze.bronze_load GROUP BY zone_code
ORDER BY source, zone_code